#Support Ticket Classifier using Hugging Face Transformers
This notebook builds an end-to-end NLP model to classify customer support tickets into categories like Billing, Technical, Delivery, etc.

In [1]:
#Import all required Python libraries
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import torch

In [2]:
# Load dataset from Hugging Face Hub
# This dataset contains customer support tickets and their categories
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")

# Convert training split into Pandas DataFrame for easier handling
train_df = pd.DataFrame(dataset['train'])
train_df.head()

README.md: 0.00B [00:00, ?B/s]

aa_dataset-tickets-multi-lang-5-2-50-ver(…):   0%|          | 0.00/26.0M [00:00<?, ?B/s]

(…)set-tickets-german_normalized_50_5_2.csv: 0.00B [00:00, ?B/s]

dataset-tickets-multi-lang-4-20k.csv:   0%|          | 0.00/18.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61765 [00:00<?, ? examples/s]

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None


In [3]:
print(train_df.columns)

Index(['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language',
       'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6',
       'tag_7', 'tag_8'],
      dtype='object')


In [4]:
# Convert categorical labels into numerical format
# Target column in this dataset is "type"

labels = train_df['type'].unique()

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

# Create numeric label column
train_df['label'] = train_df['type'].map(label2id)

In [5]:
train_df = train_df.dropna(subset=['type'])

In [6]:
labels = train_df['type'].unique()

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

train_df['label'] = train_df['type'].map(label2id)

In [7]:
# Split dataset into training and validation sets
# 80% training data, 20% validation data

train_texts, val_texts, train_labels, val_labels = train_test_split(
    (train_df['subject'].fillna('') + " " + train_df['body'].fillna('')).tolist(),
    train_df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)

In [8]:
# Load pretrained tokenizer (DistilBERT)
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize training and validation text data
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# Create PyTorch Dataset class
# This formats data so Hugging Face Trainer can use it
class TicketDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

In [10]:
#Create dataset objects for training and validation
train_dataset = TicketDataset(train_encodings, train_labels)
val_dataset = TicketDataset(val_encodings, val_labels)

In [11]:
# Load pretrained DistilBERT model for classification task
# We specify number of labels based on dataset
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# Define evaluation metrics (Accuracy and F1 Score)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }

In [13]:
#Define training configuration
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs'
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [14]:
# Initialize Hugging Face Trainer
# Trainer handles training loop, evaluation, and optimization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [15]:
#Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.412559,0.394487,0.806956,0.792226
2,0.322612,0.336357,0.846676,0.844287
3,0.237996,0.338058,0.863346,0.860599


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7290, training_loss=0.34295839014040264, metrics={'train_runtime': 1502.2248, 'train_samples_per_second': 77.623, 'train_steps_per_second': 4.853, 'total_flos': 3861794223092736.0, 'train_loss': 0.34295839014040264, 'epoch': 3.0})

In [16]:
#Evaluate the model performance
trainer.evaluate()

{'eval_loss': 0.3380582928657532,
 'eval_accuracy': 0.8633463675653427,
 'eval_f1': 0.8605992862793707,
 'eval_runtime': 32.6663,
 'eval_samples_per_second': 297.493,
 'eval_steps_per_second': 18.612,
 'epoch': 3.0}

In [30]:
import torch

def predict(text):
    model.eval()

    # move model + inputs to same device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()
    return id2label[pred]

In [31]:
for t in ["Query About Smart Home System Integration Feat", "refund request", "Inquiry about systems"]:
    print(predict(t))

Request
Incident
Request


In [40]:
tests = [
    "Payment failed twice but money deducted from account",
    "Refund has not been processed even after 7 days",
    "Charged incorrectly for subscription renewal"
]

for t in tests:
    print(f"Input: {t}")
    print(f"Predicted: {predict(t)}")
    print("-" * 50)


Input: Payment failed twice but money deducted from account
Predicted: Problem
--------------------------------------------------
Input: Refund has not been processed even after 7 days
Predicted: Incident
--------------------------------------------------
Input: Charged incorrectly for subscription renewal
Predicted: Problem
--------------------------------------------------


In [41]:
test_cases = [
    "Query About Smart Home System Integration Feat",
    "refund request",
    "Inquiry about systems"
]

for text in test_cases:
    result = predict(text)
    print(f"Input: {text}")
    print(f"Predicted Label: {result}")
    print("-" * 50)

Input: Query About Smart Home System Integration Feat
Predicted Label: Request
--------------------------------------------------
Input: refund request
Predicted Label: Incident
--------------------------------------------------
Input: Inquiry about systems
Predicted Label: Request
--------------------------------------------------
